# Sesión 6 — Modelación bajo incertidumbre, PERT y Sistemas de Espera

**Métodos y Modelos Cuantitativos**

Este notebook aplica los ejemplos de la sesión. Modifica los parámetros y vuelve a ejecutar las celdas para analizar cómo cambian los resultados.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import norm

pd.options.display.float_format = '{:,.3f}'.format

## 1. PERT — Lanzamiento de una ruta logística intermodal

Partimos de las estimaciones optimista (`a`), más probable (`m`) y pesimista (`b`) utilizadas en la presentación.

In [ ]:
actividades = pd.DataFrame({
    'Actividad': ['A','B','C','D','E'],
    'Descripcion': ['Habilitación de permisos y vías','Contratación de transportistas','Adecuación de rampa en CEDIS','Pruebas de sistemas GPS / TMS','Inspección final e inauguración'],
    'Predecesoras': [[],['A'],['A'],['B'],['C','D']],
    'a': [2,3,4,2,1], 'm': [4,5,7,3,2], 'b': [6,7,10,10,9]
})
actividades

Calculamos el tiempo esperado y la varianza PERT de cada actividad.

In [ ]:
actividades['te']=(actividades['a']+4*actividades['m']+actividades['b'])/6
actividades['varianza']=((actividades['b']-actividades['a'])/6)**2
actividades[['Actividad','Descripcion','te','varianza']]

Construimos la red AON utilizando `te` como duración y calculamos ES, EF, LS, LF y holgura.

In [ ]:
dur=dict(zip(actividades['Actividad'],actividades['te']))
pred=dict(zip(actividades['Actividad'],actividades['Predecesoras']))
G=nx.DiGraph()
for a in actividades['Actividad']:
    G.add_node(a)
    for p in pred[a]: G.add_edge(p,a)
orden=list(nx.topological_sort(G))
ES,EF={},{}
for a in orden:
    ES[a]=max([EF[p] for p in pred[a]],default=0); EF[a]=ES[a]+dur[a]
TE=max(EF.values()); LS,LF={},{}
for a in reversed(orden):
    suc=list(G.successors(a)); LF[a]=min([LS[s] for s in suc],default=TE); LS[a]=LF[a]-dur[a]
resultado_pert=pd.DataFrame({'Actividad':orden,'ES':[ES[a] for a in orden],'EF':[EF[a] for a in orden],'LS':[LS[a] for a in orden],'LF':[LF[a] for a in orden]})
resultado_pert['Holgura']=resultado_pert['LS']-resultado_pert['ES']
resultado_pert

In [ ]:
criticas=resultado_pert.loc[np.isclose(resultado_pert['Holgura'],0),'Actividad'].tolist()
print('Ruta crítica:', ' → '.join(criticas))
print(f'Duración esperada del proyecto: {TE:.2f} días')

Visualizamos la red y las duraciones esperadas.

In [ ]:
pos={'A':(0,0),'B':(1,1),'C':(1,-1),'D':(2,1),'E':(3,0)}
plt.figure(figsize=(10,5))
nx.draw_networkx_nodes(G,pos,node_size=[2800 if n in criticas else 2400 for n in G.nodes()])
nx.draw_networkx_edges(G,pos,arrows=True,arrowsize=20,width=1.8)
nx.draw_networkx_labels(G,pos,labels={n:f'{n}\nte={dur[n]:.1f}' for n in G.nodes()},font_size=10)
plt.title('Red AON — PERT'); plt.axis('off'); plt.show()

Con la ruta crítica obtenemos la varianza y desviación estándar del proyecto. Después evaluamos fechas objetivo.

In [ ]:
var_dict=dict(zip(actividades['Actividad'],actividades['varianza']))
var_proyecto=sum(var_dict[a] for a in criticas); sigma_proyecto=math.sqrt(var_proyecto)
print(f'TE = {TE:.2f} días')
print(f'Varianza = {var_proyecto:.3f} días²')
print(f'Desviación estándar = {sigma_proyecto:.3f} días')

In [ ]:
riesgo=[]
for TD in [15,20]:
    z=(TD-TE)/sigma_proyecto; prob=norm.cdf(z); riesgo.append([TD,z,prob,prob*100])
pd.DataFrame(riesgo,columns=['Fecha objetivo (días)','Z','Probabilidad','Probabilidad (%)'])

Modifica `TD` para observar la probabilidad acumulada de terminar antes de la fecha objetivo.

In [ ]:
TD=20
x=np.linspace(TE-4*sigma_proyecto,TE+4*sigma_proyecto,500); y=norm.pdf(x,loc=TE,scale=sigma_proyecto)
plt.figure(figsize=(10,5)); plt.plot(x,y); mask=x<=TD; plt.fill_between(x[mask],y[mask],alpha=.3)
plt.axvline(TE,linestyle='--',label=f'TE = {TE:.1f}'); plt.axvline(TD,linestyle='--',label=f'TD = {TD}')
plt.xlabel('Duración del proyecto (días)'); plt.ylabel('Densidad'); plt.title(f'P(T ≤ {TD}) = {norm.cdf((TD-TE)/sigma_proyecto):.2%}'); plt.legend(); plt.show()

### Actividad PERT

Modifica al menos una estimación `a`, `m` o `b`. Ejecuta nuevamente el análisis e identifica si cambia la ruta crítica, la duración esperada o la probabilidad de cumplir una fecha objetivo. Escribe una breve interpretación.

## 2. Teoría de Colas — M/M/1

Creamos una función para calcular los principales indicadores de una fila con un servidor.

In [ ]:
def mm1(lam,mu):
    if lam>=mu: return {'rho':lam/mu,'estado':'Inestable','P0':np.nan,'Pw':np.nan,'Lq':np.inf,'L':np.inf,'Wq':np.inf,'W':np.inf}
    rho=lam/mu; P0=1-rho; Lq=rho**2/(1-rho); L=rho/(1-rho); Wq=Lq/lam; W=L/lam
    return {'rho':rho,'estado':'Estable','P0':P0,'Pw':rho,'Lq':Lq,'L':L,'Wq':Wq,'W':W}
pd.Series(mm1(6,4))

Con 6 llegadas por hora y una capacidad de 4 camiones/hora, una sola rampa es insuficiente: el sistema es inestable.

## 3. M/M/c — Varias rampas

Ahora utilizamos una fila común y `c` servidores idénticos. La función incluye P0, Erlang C (Pw), Lq, L, Wq y W.

In [ ]:
def mmc(lam,mu,c):
    rho=lam/(c*mu)
    if rho>=1: return {'c':c,'rho':rho,'estado':'Inestable','P0':np.nan,'Pw':np.nan,'Lq':np.inf,'L':np.inf,'Wq_h':np.inf,'W_h':np.inf}
    a=lam/mu
    suma=sum((a**n)/math.factorial(n) for n in range(c))
    termino=(a**c/math.factorial(c))*(1/(1-rho)); P0=1/(suma+termino); Pw=termino*P0
    Lq=Pw*rho/(1-rho); Wq=Lq/lam; W=Wq+1/mu; L=lam*W
    return {'c':c,'rho':rho,'estado':'Estable','P0':P0,'Pw':Pw,'Lq':Lq,'L':L,'Wq_h':Wq,'W_h':W}


Reproducimos el ejemplo de la presentación: 6 camiones/hora y 15 minutos de servicio por camión, es decir, μ = 4 camiones/hora por rampa.

In [ ]:
lam=6; mu=4
resultados=[mmc(lam,mu,c) for c in [1,2,3,4]]
tabla_colas=pd.DataFrame(resultados)
tabla_colas['rho (%)']=tabla_colas['rho']*100; tabla_colas['P0 (%)']=tabla_colas['P0']*100; tabla_colas['Pw (%)']=tabla_colas['Pw']*100
tabla_colas['Wq (min)']=tabla_colas['Wq_h']*60; tabla_colas['W (min)']=tabla_colas['W_h']*60
tabla_colas[['c','estado','rho (%)','P0 (%)','Pw (%)','Lq','L','Wq (min)','W (min)']]

Comparamos cómo cambia la espera al aumentar el número de rampas.

In [ ]:
estables=tabla_colas.replace([np.inf,-np.inf],np.nan).dropna(subset=['Wq_h'])
plt.figure(figsize=(9,5)); plt.plot(estables['c'],estables['Wq (min)'],marker='o'); plt.xticks(estables['c']); plt.xlabel('Número de rampas'); plt.ylabel('Espera promedio Wq (min)'); plt.title('Efecto de la capacidad sobre el tiempo de espera'); plt.grid(alpha=.25); plt.show()

## 4. Ley de Little

Verificamos las relaciones L = λW y Lq = λWq para el caso de tres rampas.

In [ ]:
r3=mmc(lam,mu,3)
print(f"L = {r3['L']:.3f} | λW = {lam*r3['W_h']:.3f}")
print(f"Lq = {r3['Lq']:.3f} | λWq = {lam*r3['Wq_h']:.3f}")

## 5. Costo total por hora

Evaluamos el equilibrio entre el costo de operar rampas y el costo generado por los camiones que permanecen esperando.

In [ ]:
Cs=400; Cw=1000
costos=[]
for c in range(1,7):
    r=mmc(lam,mu,c); cs=c*Cs
    ce=r['Lq']*Cw if r['estado']=='Estable' else np.nan; ct=cs+ce if r['estado']=='Estable' else np.nan
    costos.append({'Rampas':c,'Costo servicio/h':cs,'Lq':r['Lq'],'Costo espera/h':ce,'Costo total/h':ct})
tabla_costos=pd.DataFrame(costos); tabla_costos

In [ ]:
factibles=tabla_costos.dropna(subset=['Costo total/h']); mejor=factibles.loc[factibles['Costo total/h'].idxmin()]
print(f"Alternativa de menor costo entre las evaluadas: {int(mejor['Rampas'])} rampas")
print(f"Costo total: ${mejor['Costo total/h']:,.2f} por hora")

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(factibles['Rampas'],factibles['Costo servicio/h'],marker='o',label='Costo de servicio')
plt.plot(factibles['Rampas'],factibles['Costo espera/h'],marker='o',label='Costo de espera')
plt.plot(factibles['Rampas'],factibles['Costo total/h'],marker='o',linewidth=2,label='Costo total')
plt.xticks(factibles['Rampas']); plt.xlabel('Número de rampas'); plt.ylabel('Costo por hora ($)'); plt.title('Capacidad de servicio vs. costo total'); plt.legend(); plt.grid(alpha=.25); plt.show()

### Actividad de Teoría de Colas

Analiza un sistema logístico distinto. Define una tasa de llegada, un tiempo promedio de servicio y al menos tres alternativas de número de servidores. Compara utilización, probabilidad de espera, Lq, Wq y costo total.

**Entrega:** tabla de resultados, una gráfica y una recomendación breve sobre cuántos servidores deberían operar y por qué.

## 6. Actividad integradora

Selecciona **uno** de los dos bloques:

**Opción A — PERT:** modifica el caso de la ruta logística, evalúa al menos dos fechas objetivo y explica el riesgo de cumplimiento.

**Opción B — Colas:** plantea un sistema de servicio logístico, compara diferentes niveles de capacidad y recomienda una alternativa considerando desempeño y costo.

No basta con mostrar los resultados del código: la conclusión debe explicar qué decisión tomarías a partir de ellos.